This data cleaning script eliminates duplicates, removes low-quality data, and exports a structured dataset that can be used to support sentiment analysis down the line.


In [2]:
import pandas as pd
import re

input_file = "updated_sony_raw_data.csv"
output_file = "sony_cleaned_data.csv"

In [3]:
df = pd.read_csv(input_file)

print("Shape of raw data:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

Shape of raw data: (363, 5)

Columns:
['product', 'source', 'url', 'paragraph_id', 'raw_text']

First 5 rows:
             product     source                                 url  \
0  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
1  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
2  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
3  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
4  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   

   paragraph_id                                           raw_text  
0             2  VAIO Corporation ( VAIO 株式会社 , Baio Kabushiki ...  
1             3  Vaio began as a brand of Sony , [ 5 ] introduc...  
2             4  In 2025, JIP completed the sale of its 91.40% ...  
3             5  Originally an acronym of "Video Audio Input Ou...  
4             6  As of 2023, Vaio is operational in the followi...  


In [4]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("\nStandardized columns:")
print(df.columns.tolist())


Standardized columns:
['product', 'source', 'url', 'paragraph_id', 'raw_text']


In [5]:
text_columns = ["product", "source", "url", "paragraph_id", "raw_text"]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [6]:
product_map = {
    "sony vaio laptops": "Sony VAIO Laptops",
    "sony playstation vue": "Sony PlayStation Vue",
    "sony playmemories online": "Sony PlayMemories Online"
}

df["product"] = df["product"].str.lower().map(product_map).fillna(df["product"])

In [7]:
df["source"] = df["source"].str.strip()

source_map = {
    "wikipedia": "Wikipedia",
    "the verge": "The Verge",
    "sony support": "Sony Support",
    "sony help guide": "Sony Help Guide",
    "britannica laptop computer": "Britannica Laptop Computer",
    "britannica streaming media": "Britannica Streaming Media",
    "techradar": "Techradar",
    "dpreview": "DP Review"
}

df["source"] = df["source"].str.lower().map(source_map).fillna(df["source"])

In [8]:

df["url"] = df["url"].str.strip().str.lower()
def clean_text(text):
    text = str(text)

    # Remove line breaks and tabs
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    # Remove citation-style bracket artifacts like [ 5 ] or [5]
    text = re.sub(r"\[\s*\d+\s*\]", "", text)

    # Remove extra spaces again after bracket cleanup
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_text"] = df["raw_text"].apply(clean_text)

In [9]:
df = df[df["clean_text"].notna()]
df = df[df["clean_text"].str.len() > 20]

print("\nShape after removing weak text rows:", df.shape)


Shape after removing weak text rows: (363, 6)


In [10]:
df["char_count"] = df["clean_text"].str.len()
df["word_count"] = df["clean_text"].str.split().str.len()

print("\nSummary of text length features:")
print(df[["char_count", "word_count"]].describe())


Summary of text length features:
         char_count   word_count
count    363.000000   363.000000
mean     906.556474   165.250689
std     3257.586924   611.773349
min       51.000000     6.000000
25%       95.000000    16.000000
50%      305.000000    51.000000
75%      718.500000   126.000000
max    33716.000000  6108.000000


In [11]:
before_dedup = df.shape[0]

df = df.drop_duplicates(subset=["product", "source", "url", "clean_text"])

after_dedup = df.shape[0]

print(f"\nDuplicates removed: {before_dedup - after_dedup}")
print("Shape after deduplication:", df.shape)


Duplicates removed: 0
Shape after deduplication: (363, 8)


In [12]:
df["is_short_text"] = df["word_count"] < 15

df["is_support_source"] = df["source"].isin(["Sony Support", "Sony Help Guide"])

print("\nFlag counts:")
print("Short text rows:", df["is_short_text"].sum())
print("Support/help rows:", df["is_support_source"].sum())


Flag counts:
Short text rows: 78
Support/help rows: 14


In [13]:
df["paragraph_id"] = df["paragraph_id"].astype(str).str.strip()

In [14]:
df["is_short_text"] = df["word_count"] < 15
df["is_support_source"] = df["source"].isin(["Sony Support", "Sony Help Guide"])

final_columns = [
    "product",
    "source",
    "url",
    "paragraph_id",
    "raw_text",
    "clean_text",
    "char_count",
    "word_count",
    "is_short_text",
    "is_support_source"
]

df = df[final_columns]

In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load a pre-trained sentence transformer model
# 'all-MiniLM-L6-v2' is a good balance of performance and speed
model = SentenceTransformer('all-MiniLM-L6-v2')

# Get the 'DP Review' dataframe again
dpreview_df = df[df['source'] == 'DP Review'].copy()

# Generate embeddings for the clean_text column
print("Generating embeddings for DP Review texts...")
dpreview_embeddings = model.encode(dpreview_df['clean_text'].tolist(), show_progress_bar=True)
print("Embeddings generated.")

# Calculate cosine similarity between all pairs of embeddings
similarity_matrix = cosine_similarity(dpreview_embeddings)

# Identify highly similar pairs
# We'll set a high threshold, e.g., 0.95, to be conservative
high_similarity_threshold = 0.95

similar_indices = np.where(similarity_matrix > high_similarity_threshold)

# Exclude self-similarity (where index_i == index_j)
similar_pairs = []
for i, j in zip(similar_indices[0], similar_indices[1]):
    if i < j:  # Ensure unique pairs and avoid duplicates (e.g., (1,2) and (2,1))
        similar_pairs.append((i, j))

print(f"Found {len(similar_pairs)} pairs of highly similar texts in DP Review (similarity > {high_similarity_threshold}).")

# Prepare to identify rows for removal
indices_to_remove = set()
for i, j in similar_pairs:
    text_i = dpreview_df.iloc[i]['clean_text']
    text_j = dpreview_df.iloc[j]['clean_text']

    # Decide which one to remove. A simple strategy: remove the shorter text
    # If lengths are equal, remove the one with higher index (appeared later)
    if len(text_i) >= len(text_j):
        indices_to_remove.add(j)
    else:
        indices_to_remove.add(i)

# Convert to actual DataFrame indices
df_indices_to_remove = dpreview_df.iloc[list(indices_to_remove)].index.tolist()

print(f"\nNumber of DP Review rows identified for removal based on high similarity: {len(df_indices_to_remove)}")
print(f"Target reduction for DP Review (10% of {dpreview_df.shape[0]}): {int(dpreview_df.shape[0] * 0.1)}")

# Display some of the original text for removed rows to allow user review
if len(df_indices_to_remove) > 0:
    print("\nSample of 'DP Review' rows proposed for removal (original 'clean_text'):")
    for idx in df_indices_to_remove[:5]: # Display up to 5 samples
        print(f"- Index {idx}: {df.loc[idx, 'clean_text'][:100]}...") # Show first 100 chars

# Store the indices for potential later use if the user approves
# Note: This step just identifies, does not modify the original df yet.
identified_similar_dpreview_indices = df_indices_to_remove

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for DP Review texts...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings generated.
Found 113 pairs of highly similar texts in DP Review (similarity > 0.95).

Number of DP Review rows identified for removal based on high similarity: 61
Target reduction for DP Review (10% of 216): 21

Sample of 'DP Review' rows proposed for removal (original 'clean_text'):
- Index 238: And guess what, after I tried so hard to finally spent my $30 or so and got all the apps I want from...
- Index 244: kiri said: Sony might make some nice hardware.. but their online playmemories service is atrocious. ...
- Index 117: Viltrox is trying out a new kind of optical design GoPro reveals the pricing for its most exciting c...
- Index 248: Apr 11, 2016 #13 I've seen several threads over the past months with the same general theme, that pl...
- Index 249: I've seen several threads over the past months with the same general theme, that playmemories doesn'...


In [16]:
# Remove the identified similar DP Review rows from the main DataFrame
df = df.drop(identified_similar_dpreview_indices)

print(f"\nRemoved {len(identified_similar_dpreview_indices)} highly similar DP Review rows.")
print("New shape of DataFrame:", df.shape)

# Verify the number of DP Review rows after removal
dpreview_df_after_removal = df[df['source'] == 'DP Review']
print(f"DP Review rows remaining: {dpreview_df_after_removal.shape[0]}")


Removed 61 highly similar DP Review rows.
New shape of DataFrame: (302, 10)
DP Review rows remaining: 155


In [17]:
df.to_csv(output_file, index=False)

print(f"\nCleaned file saved as: {output_file}")
print("Final shape:", df.shape)
print("\nFinal preview:")
print(df.head())


Cleaned file saved as: sony_cleaned_data.csv
Final shape: (302, 10)

Final preview:
             product     source                                 url  \
0  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
1  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
2  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
3  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
4  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   

  paragraph_id                                           raw_text  \
0            2  VAIO Corporation ( VAIO 株式会社 , Baio Kabushiki ...   
1            3  Vaio began as a brand of Sony , [ 5 ] introduc...   
2            4  In 2025, JIP completed the sale of its 91.40% ...   
3            5  Originally an acronym of "Video Audio Input Ou...   
4            6  As of 2023, Vaio is operational in the followi...   

                                          clean_text  char_count  word_co

In [18]:
print("Unique values in 'source' column:")
print(df['source'].unique())

Unique values in 'source' column:
['Wikipedia' 'Britannica Laptop Computer' 'Techradar' 'Sony Support'
 'Sony Help Guide' 'DP Review' 'Britannica Streaming Media' 'The Verge']


In [19]:
dpreview_df = df[df['source'] == 'DP Review']

print(f"Total rows from DP Review: {dpreview_df.shape[0]}")
print(f"DP Review rows with short text: {dpreview_df['is_short_text'].sum()}")
print(f"DP Review rows from support/help sources: {dpreview_df['is_support_source'].sum()}")

Total rows from DP Review: 155
DP Review rows with short text: 50
DP Review rows from support/help sources: 0


In [1]:
# Install sentence-transformers for semantic similarity
!pip install sentence-transformers